**API FUNDAMENTALS**


**AUTHENTICATED REQUESTS,JSON PARSING,RATE LIMIT AWARENESS**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import datetime as date
import seaborn as sns
import requests
import yfinance as yf
import pandas as pd

In [2]:
url = "https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=IKCVDTFY1DGY0J1J"

In [3]:
response = requests.get(url)

In [4]:
print(response.json())

{'Meta Data': {'1. Information': 'Daily Prices (open, high, low, close) and Volumes', '2. Symbol': 'IBM', '3. Last Refreshed': '2026-06-24', '4. Output Size': 'Compact', '5. Time Zone': 'US/Eastern'}, 'Time Series (Daily)': {'2026-06-24': {'1. open': '261.8430', '2. high': '265.0700', '3. low': '256.1836', '4. close': '262.9600', '5. volume': '8149935'}, '2026-06-23': {'1. open': '261.5800', '2. high': '267.5300', '3. low': '255.2600', '4. close': '264.9400', '5. volume': '15686117'}, '2026-06-22': {'1. open': '248.4300', '2. high': '253.3100', '3. low': '243.8100', '4. close': '252.2200', '5. volume': '9615393'}, '2026-06-18': {'1. open': '251.3800', '2. high': '252.4700', '3. low': '243.6800', '4. close': '249.1000', '5. volume': '16619749'}, '2026-06-17': {'1. open': '266.3500', '2. high': '268.8700', '3. low': '261.8800', '4. close': '262.3500', '5. volume': '5537042'}, '2026-06-16': {'1. open': '270.8700', '2. high': '276.6200', '3. low': '268.6100', '4. close': '270.8100', '5. vo

In [5]:
data = response.json()
time_series = data['Time Series (Daily)']

dates = []
closes = []

for date, values in time_series.items():
    dates.append(date)
    closes.append(float(values['4. close']))

closing_df = pd.DataFrame({'date': dates, 'close': closes})
closing_df['date'] = pd.to_datetime(closing_df['date'])
closing_df = closing_df.sort_values('date').reset_index(drop=True)
print(closing_df.head())

        date   close
0 2026-01-30  306.70
1 2026-02-02  314.73
2 2026-02-03  294.31
3 2026-02-04  289.05
4 2026-02-05  289.89


In [6]:
full_df = pd.DataFrame.from_dict(time_series, orient='index')
full_df.index = pd.to_datetime(full_df.index)
full_df = full_df.sort_index()
full_df.columns = ['open', 'high', 'low', 'close', 'volume']
full_df = full_df.astype(float)
print(full_df.head())

              open     high     low   close      volume
2026-01-30  307.60  307.783  299.73  306.70   5940669.0
2026-02-02  307.51  316.640  306.41  314.73   4581189.0
2026-02-03  312.40  312.975  283.85  294.31  11296020.0
2026-02-04  291.41  291.410  278.96  289.05   8708033.0
2026-02-05  286.10  291.810  285.10  289.89   5532797.0


**PIPLELINE**

In [7]:
tickers = ['RELIANCE.NS','TCS.NS','INFY.NS']

In [8]:
data = yf.download(tickers, period='5d')
latest_prices = data['Close'].iloc[-1]
print(latest_prices)

/tmp/ipykernel_2454/1571182665.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, period='5d')
[*********************100%***********************]  3 of 3 completed

Ticker
INFY.NS        1041.199951
RELIANCE.NS    1318.099976
TCS.NS         2094.699951
Name: 2026-06-25 00:00:00, dtype: float64


In [9]:
today_date = latest_prices.name.date()  # extract just the date part

row = pd.DataFrame({
    'date': [today_date],
    'RELIANCE.NS': [latest_prices['RELIANCE.NS']],
    'TCS.NS': [latest_prices['TCS.NS']],
    'INFY.NS': [latest_prices['INFY.NS']]
})

print(row)

         date  RELIANCE.NS       TCS.NS      INFY.NS
0  2026-06-25  1318.099976  2094.699951  1041.199951


In [10]:
import os

filename = 'stock_pipeline.csv'

if os.path.exists(filename):
    existing = pd.read_csv(filename)
    existing['date'] = pd.to_datetime(existing['date']).dt.date

    if today_date in existing['date'].values:
        print(f"Already have data for {today_date} — skipping.")
    else:
        existing = pd.concat([existing, row], ignore_index=True)
        existing.to_csv(filename, index=False)
        print(f"Appended new row for {today_date}.")
else:
    row.to_csv(filename, index=False)
    print(f"Created new file with first row for {today_date}.")

Created new file with first row for 2026-06-25.


In [11]:
pd.read_csv('stock_pipeline.csv')

,date,RELIANCE.NS,TCS.NS,INFY.NS
0,2026-06-25,1318.099976,2094.699951,1041.199951
